# Campus GAT V23 - Cluster Trunk Roads


In [ ]:
import os
!git -C swe3032 pull 2>/dev/null || git clone https://github.com/SKKUAIProjectTeam1/swe3032.git
os.chdir('swe3032')

In [8]:
import glob, os
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from scipy.ndimage import distance_transform_edt, maximum_filter, gaussian_filter

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


Tesla T4


In [9]:
# ── 설정 ──────────────────────────────────────────────────────────────────────
RES           = 100
N             = RES * RES
NODE_DIM      = 5
GAT_HEADS     = 4
DIFF_STEPS    = 25
EPOCHS        = 300
LR            = 3e-4
WARMUP_EPOCHS = 50
ATTN_DROPOUT  = 0.1
ACCUM_STEPS   = 10
N_GATES       = 2
GATE_MIN_DIST = 14
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# V23 핵심 변경점
# 1) 건물 하나하나를 차도 목적지로 보지 않고 가까운 건물군을 cluster로 묶음
# 2) 건물 외곽 buffer가 아니라 빈 공간의 medial ridge를 도로 후보로 사용
# 3) 차도는 gate → cluster 대표점 연결을 목표로 함
# 4) 각 건물은 차도가 직접 붙지 않아도 근처 접근 가능하면 통과
CLUSTER_EPS     = 13.0     # grid cell 기준. 크게 하면 더 많은 건물이 한 묶음이 됨
MIN_CLEARANCE   = 3.0      # 건물에서 최소 이 정도 떨어진 곳만 trunk 후보로 봄
RIDGE_FILTER    = 7        # medial ridge local maximum filter size
RIDGE_SIGMA     = 1.2      # ridge score smoothing
MAX_WALK_DIST   = 8.0      # 건물에서 차도까지 허용 보행 접근 거리

IMG_DIR = 'collegemap/images'
TXT_DIR = 'collegemap/txt'
OUT_DIR = 'output'
CKPT    = os.path.join(OUT_DIR, 'v23_cluster_roads.pth')
os.makedirs(OUT_DIR, exist_ok=True)


In [10]:
# ── 고정 그래프 토폴로지 ───────────────────────────────────────────────────────
def _build_edges(res):
    edges = []
    for y in range(res):
        for x in range(res):
            s = y * res + x
            for dy in (-1, 0, 1):
                for dx in (-1, 0, 1):
                    if dy == 0 and dx == 0: continue
                    nx_, ny_ = x + dx, y + dy
                    if 0 <= nx_ < res and 0 <= ny_ < res:
                        edges.append((s, ny_ * res + nx_))
    edges += [(i, i) for i in range(res * res)]
    return torch.tensor(edges, dtype=torch.long).t().contiguous()

EDGE_INDEX = _build_edges(RES).to(DEVICE)

_bdy = []
for i in range(10, RES - 10):
    _bdy += [i, (RES-1)*RES + i, i*RES, i*RES + (RES-1)]
BOUNDARY   = sorted(set(_bdy))
BOUNDARY_T = torch.tensor(BOUNDARY, dtype=torch.long).to(DEVICE)

In [11]:
# ── 데이터 로딩 ────────────────────────────────────────────────────────────────
def _find_txt(img_path):
    stem   = os.path.basename(img_path).replace('_building_mask.png', '')
    direct = os.path.join(TXT_DIR, stem + '_building_places.txt')
    if os.path.exists(direct): return direct
    prefix = stem.replace('-', '_').split('_')[0]
    for fn in os.listdir(TXT_DIR):
        if fn.endswith('_building_places.txt') and fn.startswith(prefix):
            return os.path.join(TXT_DIR, fn)
    return None

def _nearest_road_node(cy, cx, is_bld_grid):
    non_bld_yx = np.argwhere(is_bld_grid == 0)
    if len(non_bld_yx) == 0: return int(cy) * RES + int(cx)
    dists = (non_bld_yx[:,0] - cy)**2 + (non_bld_yx[:,1] - cx)**2
    best  = non_bld_yx[np.argmin(dists)]
    return int(best[0]) * RES + int(best[1])

def _nearest_ridge_node(cy, cx, is_bld_grid, ridge_grid, min_score=0.05):
    # 클러스터 대표점은 건물 외곽이 아니라 공통 도로 후보선 위에 잡는다.
    cand = np.argwhere((is_bld_grid == 0) & (ridge_grid > min_score))
    if len(cand) == 0:
        return _nearest_road_node(cy, cx, is_bld_grid)
    dists = (cand[:,0] - cy)**2 + (cand[:,1] - cx)**2
    best  = cand[np.argmin(dists)]
    return int(best[0]) * RES + int(best[1])

def _cluster_centers(centers, eps=CLUSTER_EPS):
    # 단순 connected-component clustering.
    # 거리 eps 안에 있는 건물들은 같은 건물군으로 묶는다.
    n = len(centers)
    visited = [False] * n
    clusters = []
    for i in range(n):
        if visited[i]:
            continue
        stack = [i]
        visited[i] = True
        group = []
        while stack:
            u = stack.pop()
            group.append(u)
            uy, ux = centers[u]
            for v in range(n):
                if visited[v]:
                    continue
                vy, vx = centers[v]
                if ((uy - vy)**2 + (ux - vx)**2)**0.5 <= eps:
                    visited[v] = True
                    stack.append(v)
        clusters.append(group)
    return clusters

def _make_medial_ridge(is_bld):
    # 기존 V22-B의 (dist > 1.5) & (dist < 7.0)은 건물 외곽 buffer라서
    # 모든 건물 주변을 둘러싸는 도로를 만들기 쉬웠음.
    # V23은 free space의 local maximum을 사용해 공통 trunk 후보를 만든다.
    free = (is_bld == 0)
    dist = distance_transform_edt(free)
    clear = free & (dist >= MIN_CLEARANCE)
    local_max = (dist >= maximum_filter(dist, size=RIDGE_FILTER) - 1e-6)
    ridge = (clear & local_max).astype(np.float32)
    ridge_score = gaussian_filter(ridge, sigma=RIDGE_SIGMA)
    if ridge_score.max() > 0:
        ridge_score = ridge_score / (ridge_score.max() + 1e-6)
    # 너무 넓게 번지는 것을 방지
    ridge_score = (ridge_score * clear.astype(np.float32)).astype(np.float32)
    return ridge_score, dist.astype(np.float32)

def load_campus(img_path, txt_path):
    img    = Image.open(img_path).convert('L')
    W, H   = img.size
    is_bld = (np.array(img.resize((RES,RES), resample=Image.NEAREST)) > 128).astype(np.float32)
    ridge_score, dist = _make_medial_ridge(is_bld)
    dist_n = (dist / (dist.max() + 1e-6)).astype(np.float32)
    xs = np.tile(np.arange(RES), RES).astype(np.float32) / RES
    ys = np.repeat(np.arange(RES), RES).astype(np.float32) / RES
    node_feats = np.stack([is_bld.flatten(), ridge_score.flatten(), dist_n.flatten(), xs, ys], axis=1)

    ib_flat  = torch.tensor(is_bld.flatten(), dtype=torch.float32).to(DEVICE)
    src, dst = EDGE_INDEX[0], EDGE_INDEX[1]
    bld_mask = (ib_flat[src] > 0) | (ib_flat[dst] > 0)

    ns = {}
    exec(open(txt_path, encoding='utf-8').read(), ns)
    poly = ns['BUILDING_POLY']

    centers = []
    for pts in poly.values():
        cy = np.mean([p[1] for p in pts]) * RES / H
        cx = np.mean([p[0] for p in pts]) * RES / W
        centers.append((cy, cx))

    # 건물별 접근 확인용 node. 차도 연결 대상은 아님.
    bld_nodes = list({_nearest_road_node(cy, cx, is_bld) for cy, cx in centers})

    # 차도 연결 대상은 건물군 대표점으로 축소.
    clusters = _cluster_centers(centers, eps=CLUSTER_EPS)
    cluster_nodes = []
    cluster_members = []
    for group in clusters:
        cy = np.mean([centers[i][0] for i in group])
        cx = np.mean([centers[i][1] for i in group])
        cluster_nodes.append(_nearest_ridge_node(cy, cx, is_bld, ridge_score))
        cluster_members.append(group)

    return {
        'node_feats':      torch.tensor(node_feats, dtype=torch.float32).to(DEVICE),
        'is_building':     ib_flat,
        'bld_mask':        bld_mask,
        'ridge':           torch.tensor(ridge_score.flatten(), dtype=torch.float32).to(DEVICE),
        'building_nodes':  torch.tensor(bld_nodes, dtype=torch.long).to(DEVICE),
        'cluster_nodes':   torch.tensor(cluster_nodes, dtype=torch.long).to(DEVICE),
        'cluster_members': cluster_members,
        'poly': poly,
        'name': os.path.basename(img_path).replace('_building_mask.png',''),
        'W': W,
        'H': H,
    }

campuses = []
for img_path in sorted(glob.glob(os.path.join(IMG_DIR, '*_building_mask.png'))):
    txt = _find_txt(img_path)
    if txt:
        try:
            campuses.append(load_campus(img_path, txt))
        except Exception as e:
            print(f'skip {os.path.basename(img_path)}: {e}')
print(f'{len(campuses)}개 캠퍼스 로드 완료')
for c in campuses:
    print(f'{c["name"]}: buildings={len(c["building_nodes"])} clusters={len(c["cluster_nodes"])}')


0개 캠퍼스 로드 완료


In [ ]:
# ── 모델 ──────────────────────────────────────────────────────────────────────
class MHGATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, heads=4, dropout=0.0):
        super().__init__()
        assert out_dim % heads == 0
        self.heads = heads; self.dh = out_dim // heads
        self.W    = nn.Linear(in_dim, out_dim, bias=False)
        self.a_s  = nn.Parameter(torch.empty(heads, self.dh))
        self.a_d  = nn.Parameter(torch.empty(heads, self.dh))
        self.norm = nn.LayerNorm(out_dim)
        self.skip = nn.Linear(in_dim, out_dim, bias=False) if in_dim != out_dim else nn.Identity()
        self.drop = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.W.weight, gain=1.414)
        nn.init.xavier_normal_(self.a_s.unsqueeze(0)); nn.init.xavier_normal_(self.a_d.unsqueeze(0))

    def forward(self, x, ei):
        n = x.size(0); src, dst = ei[0], ei[1]
        h = self.W(x).view(n, self.heads, self.dh)
        e = F.leaky_relu((h[src]*self.a_s).sum(-1) + (h[dst]*self.a_d).sum(-1), 0.2)
        e_exp = torch.exp(e - e.max())
        denom = torch.zeros(n, self.heads, device=x.device)
        denom.scatter_add_(0, src.unsqueeze(1).expand(-1, self.heads), e_exp)
        alpha = self.drop(e_exp / (denom[src] + 1e-8))
        msg   = alpha.unsqueeze(-1) * h[dst]
        agg   = torch.zeros(n, self.heads, self.dh, device=x.device)
        agg.scatter_add_(0, src[:,None,None].expand_as(msg), msg)
        return self.norm(F.elu(agg.view(n,-1)) + self.skip(x))

class CampusGAT(nn.Module):
    def __init__(self):
        super().__init__()
        self.gat1 = MHGATLayer(NODE_DIM, 64, GAT_HEADS, ATTN_DROPOUT)
        self.gat2 = MHGATLayer(64, 64, GAT_HEADS, ATTN_DROPOUT)
        self.gat3 = MHGATLayer(64, 32, GAT_HEADS, ATTN_DROPOUT)
        self.edge_head = nn.Sequential(nn.Linear(128,32), nn.ReLU(), nn.Dropout(0.1), nn.Linear(32,1))
        self.gate_head = nn.Linear(32, 1)
        for m in self.modules():
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)
        # sparse 초기화: sigmoid(-2)≈0.12
        nn.init.constant_(self.edge_head[-1].bias, -2.0)
        # gate 초기화: boundary 전체 평균이 N_GATES가 되도록 맞춰 첫 step 폭발을 줄임
        p0 = max(min(N_GATES / len(BOUNDARY), 0.99), 0.01)
        nn.init.constant_(self.gate_head.bias, float(np.log(p0 / (1.0 - p0))))

    def forward(self, feats, ei, bld_mask):
        h1 = self.gat1(feats, ei); h2 = self.gat2(h1, ei); h3 = self.gat3(h2, ei)
        src, dst = ei[0], ei[1]
        ew = torch.sigmoid(self.edge_head(torch.cat([h2[src],h2[dst]],1)).squeeze(-1)) * (~bld_mask).float()
        gs = torch.sigmoid(self.gate_head(h3[BOUNDARY_T]).squeeze(-1))
        return ew, gs

model = CampusGAT().to(DEVICE)
print(f'파라미터: {sum(p.numel() for p in model.parameters()):,}개')


In [ ]:
# ── 손실 함수 ──────────────────────────────────────────────────────────────────
def _node_strength(ew):
    src, dst = EDGE_INDEX[0], EDGE_INDEX[1]
    strength = torch.zeros(N, device=DEVICE)
    strength.scatter_add_(0, src, ew)
    strength.scatter_add_(0, dst, ew)
    return strength

def _propagate_from_seeds(ew, seeds, T=DIFF_STEPS, gain=0.75, leak=0.92):
    # V22-B의 sigmoid diffusion은 edge가 없어도 signal이 자연발생할 수 있었음.
    # V23은 seed + edge message만 누적한다.
    src, dst = EDGE_INDEX[0], EDGE_INDEX[1]
    signal = seeds.clamp(0.0, 1.0)
    for _ in range(T):
        msg = torch.zeros(N, device=DEVICE)
        msg.scatter_add_(0, dst, ew * signal[src])
        signal = torch.clamp(seeds + leak * signal + gain * msg, 0.0, 1.0)
    return signal

def gate_to_cluster_loss(ew, gs, cluster_nodes, weight=1.0):
    if cluster_nodes.numel() == 0:
        return ew.sum() * 0.
    seeds = torch.zeros(N, device=DEVICE)
    seeds[BOUNDARY_T] = gs
    signal = _propagate_from_seeds(ew, seeds)
    received = signal[cluster_nodes]
    # gate에서 각 건물군 대표점까지 닿아야 함
    return F.relu(0.55 - received).mean() * weight

def cluster_pair_loss(ew, cluster_nodes, T=DIFF_STEPS, weight=1.0):
    # 건물군끼리도 하나의 trunk network 안에 들어오도록 약하게 보조
    k = cluster_nodes.size(0)
    if k <= 1:
        return ew.sum() * 0.
    src, dst = EDGE_INDEX[0], EDGE_INDEX[1]
    signals = torch.zeros(N, k, device=DEVICE)
    signals[cluster_nodes, torch.arange(k, device=DEVICE)] = 1.0
    seeds = signals.clone()
    for _ in range(T):
        msg = torch.zeros(N, k, device=DEVICE)
        msg.scatter_add_(0, dst.unsqueeze(1).expand(-1, k), ew.unsqueeze(1) * signals[src])
        signals = torch.clamp(seeds + 0.90 * signals + 0.55 * msg, 0.0, 1.0)
    received = signals[cluster_nodes]
    off_diag = 1. - torch.eye(k, device=DEVICE)
    return F.relu(0.35 - received).mul(off_diag).sum() / (k * max(k-1, 1)) * weight

def building_access_loss(ew, building_nodes, max_walk=MAX_WALK_DIST, weight=1.0):
    # 모든 건물에 차도가 붙을 필요는 없음.
    # 가까운 공통 차도에서 걸어갈 수 있으면 OK.
    if building_nodes.numel() == 0:
        return ew.sum() * 0.
    strength = _node_strength(ew)
    road_prob = torch.sigmoid(6.0 * (strength - 0.45))
    yy = (torch.arange(N, device=DEVICE) // RES).float()
    xx = (torch.arange(N, device=DEVICE) % RES).float()
    losses = []
    for node in building_nodes:
        by = (node // RES).float()
        bx = (node % RES).float()
        dist = torch.sqrt((yy - by).pow(2) + (xx - bx).pow(2) + 1e-6)
        near = torch.exp(-dist / max_walk)
        access = (road_prob * near).max()
        losses.append(F.relu(0.35 - access))
    return torch.stack(losses).mean() * weight

def deadend_loss(ew, weight=1.0):
    # 짧은 찌꺼기 지선 억제. 단 너무 강하면 필요한 접근로까지 죽으므로 약하게 둔다.
    strength = _node_strength(ew)
    road_prob = torch.sigmoid(6.0 * (strength - 0.45))
    low_degree = torch.sigmoid(6.0 * (1.30 - strength))
    return (road_prob * low_degree).mean() * weight

def gate_loss(gs, weight=1.0):
    # sum 기준 폭발 방지. count와 binary 성향만 약하게 유도.
    count = ((gs.sum() - N_GATES) / max(N_GATES, 1)).pow(2)
    binary = (gs * (1.0 - gs)).mean()
    return (count * 2.0 + binary * 0.2) * weight

def loss_fn(ew, gs, c, scale):
    src, dst = EDGE_INDEX[0], EDGE_INDEX[1]
    rs = (c['ridge'][src] + c['ridge'][dst]) * 0.5

    # off-ridge edge는 줄이고 ridge 위 edge는 상대적으로 허용
    ridge_loss = (ew * (1.0 - rs)).mean() * (scale * 4.0 + 0.2)

    # 차도 연결 목표는 building_nodes가 아니라 cluster_nodes
    gate_conn  = gate_to_cluster_loss(ew, gs, c['cluster_nodes'], weight=scale * 8.0 + 1.0)
    pair_conn  = cluster_pair_loss(ew, c['cluster_nodes'], weight=scale * 2.0)

    # 건물별로는 직접 차도가 아니라 접근성만 확인
    access     = building_access_loss(ew, c['building_nodes'], weight=2.0)

    # 불필요한 전체 edge와 지선 억제
    sparse     = ew.mean() * 2.0
    strength   = _node_strength(ew)
    degree     = F.relu(strength - 4.0).mean() * 120.0
    dead       = deadend_loss(ew, weight=2.0)
    gate       = gate_loss(gs, weight=1.0)

    total = ridge_loss + gate_conn + pair_conn + access + sparse + degree + dead + gate
    terms = {
        'ridge': ridge_loss.detach(),
        'gate_conn': gate_conn.detach(),
        'pair_conn': pair_conn.detach(),
        'access': access.detach(),
        'sparse': sparse.detach(),
        'degree': degree.detach(),
        'dead': dead.detach(),
        'gate': gate.detach(),
    }
    return total, terms


In [ ]:
# ── 학습 ──────────────────────────────────────────────────────────────────────
opt    = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sch    = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=5e-6)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
n = len(campuses); loss_curve = []

for ep in range(1, EPOCHS+1):
    model.train(); scale = min(1.0, ep/WARMUP_EPOCHS); total = 0.; opt.zero_grad()
    term_sums = {}
    for i, c in enumerate(campuses):
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            ew, gs = model(c['node_feats'], EDGE_INDEX, c['bld_mask'])
            loss, terms = loss_fn(ew, gs, c, scale)
            loss = loss / n
        scaler.scale(loss).backward(); total += loss.item() * n
        for k, v in terms.items():
            term_sums[k] = term_sums.get(k, 0.0) + float(v.item())
        if (i+1) % ACCUM_STEPS == 0 or (i+1) == n:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt); scaler.update(); opt.zero_grad()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    sch.step(); loss_curve.append(total/n)
    if ep % 25 == 0 or ep == 1:
        msg = f'[{ep:4d}/{EPOCHS}] loss={total/n:.4f} lr={sch.get_last_lr()[0]:.2e}'
        msg += ' | ' + ' '.join([f'{k}={term_sums[k]/n:.3f}' for k in sorted(term_sums.keys())])
        print(msg)

torch.save(model.state_dict(), CKPT)
print(f'저장: {CKPT}')


In [ ]:
# ── 손실 커브 & 시각화 ─────────────────────────────────────────────────────────
plt.figure(figsize=(9,3)); plt.plot(loss_curve); plt.yscale('log')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

def plot_campus(c, show_clusters=True):
    model.eval()
    with torch.no_grad():
        ew, gs = model(c['node_feats'], EDGE_INDEX, c['bld_mask'])
    ew_np=ew.cpu().numpy(); gs_np=gs.cpu().numpy(); ei=EDGE_INDEX.cpu(); ib_np=c['is_building'].cpu().numpy()
    W,H = c['W'],c['H']
    order=np.argsort(gs_np)[::-1]; chosen=[]
    for gi in order:
        gx=BOUNDARY[gi]%RES; gy=BOUNDARY[gi]//RES
        if all(abs(gx-BOUNDARY[p]%RES)+abs(gy-BOUNDARY[p]//RES)>=GATE_MIN_DIST for p in chosen):
            chosen.append(gi)
        if len(chosen)==N_GATES: break
    fig,ax=plt.subplots(figsize=(12,12)); ax.set_facecolor('#0d1117'); fig.patch.set_facecolor('#0d1117')
    for bid,pts in c['poly'].items():
        sc=[(p[0]*RES/W,p[1]*RES/H) for p in pts]
        ax.add_patch(mpatches.Polygon(sc,closed=True,facecolor='#2d3436',edgecolor='#00cec9',lw=0.8,alpha=0.9))
        ax.text(np.mean([p[0] for p in sc]),np.mean([p[1] for p in sc]),bid,color='white',ha='center',va='center',fontsize=5.5,fontweight='bold')

    # road edge 표시. 너무 촘촘하면 percentile을 99.0 정도로 올려도 됨.
    thr=np.percentile(ew_np,98.8); mxw=max(ew_np.max(),thr+1e-8)
    for i in range(len(ew_np)):
        w=ew_np[i]
        if w<thr: continue
        s,d=ei[0,i].item(),ei[1,i].item()
        if ib_np[s]>0 or ib_np[d]>0: continue
        y1,x1=divmod(s,RES); y2,x2=divmod(d,RES)
        ax.plot([x1,x2],[y1,y2],color='#ffd32a',alpha=0.85,lw=(w-thr)/(mxw-thr)*8+1,solid_capstyle='round')

    if show_clusters:
        cn = c['cluster_nodes'].detach().cpu().numpy()
        for idx, node in enumerate(cn):
            y, x = divmod(int(node), RES)
            ax.scatter(x, y, s=60, c='#a29bfe', marker='o', edgecolors='white', lw=0.5, zorder=9)
            ax.text(x, y-1.2, f'C{idx}', color='white', ha='center', fontsize=6)

    for k,gi in enumerate(chosen):
        node=BOUNDARY[gi]; gx,gy=node%RES,node//RES
        ax.scatter(gx,gy,s=500,c='#ff3838',marker='*',edgecolors='white',lw=1.5,zorder=10)
        ax.text(gx,gy-2.5,f'GATE {chr(65+k)}',color='white',ha='center',fontsize=9,fontweight='bold',bbox=dict(fc='#d63031',alpha=0.8,ec='none',pad=1))
    ax.set_xlim(0,RES); ax.set_ylim(RES,0); ax.axis('off')
    ax.set_title(f'V23 · Cluster Trunk Roads · {c["name"].replace("_"," ").title()}',color='white',fontsize=13,pad=10)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR,f'v23_cluster_roads_{c["name"]}.png'),dpi=150,bbox_inches='tight',facecolor='#0d1117')
    plt.show(); plt.close()

# 특정 캠퍼스 확인
TARGET = 'sungkyunkwan_university'
target = next((c for c in campuses if TARGET in c['name']), None)
if target: plot_campus(target)
